# Lunar Lander VLM Verifier

Notebook scaffold for analyzing LunarLander-v3 landing attempts (GIF clips) with a multimodal LLM such as Phi-3-Vision served through Ollama. The pipeline:

1. Load and sample frames from an input GIF.
2. Build a qualitative prompt containing the six verification aspects (A–F).
3. Send images + instructions to the VLM.
4. Parse yes/no judgments and rationales for downstream consumption.


## Requirements

- Python dependencies: `imageio`, `Pillow`, `requests`, `numpy`.
- Ollama running locally with the `llava-phi3` model pulled (`ollama pull llava-phi3`).
- GIF clips for each episode stored locally (one per verification run).


In [1]:
import sys
print(sys.executable)

/Users/kevinsohn/miniconda3/bin/python


In [2]:
from __future__ import annotations

import base64
import json
import textwrap
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, List, Sequence

import imageio.v2 as imageio
import numpy as np
import requests
from PIL import Image



In [3]:
ASPECTS = [
    {
        "id": "A",
        "title": "Stable Touchdown",
        "description": "At the end of the GIF, is the lander resting stably and upright on the landing pad, and does it appear stationary?",
    },
    {
        "id": "B",
        "title": "Vertical Corridor",
        "description": "While high in the air, does the lander stay roughly centered over the pad?",
    },
    {
        "id": "C",
        "title": "Controlled Descent",
        "description": "Is the main engine (visible flame) used to actively slow the lander's descent, or does it appear to be in a free-fall?",
    },
    {
        "id": "D",
        "title": "Pitch Trimming",
        "description": "When the lander tilts, do the small side thrusters fire to correct its angle?",
    },
    {
        "id": "E",
        "title": "Pre-Landing Hover",
        "description": "Just before touching down, does the lander appear to hover and stabilize, or does it crash straight down?",
    },
    {
        "id": "F",
        "title": "Bounce Recovery",
        "description": "If the lander bounces, does the main engine fire immediately to recover?",
    },
]

OLLAMA_MODEL = "llava-phi3"
OLLAMA_URL = "http://localhost:11434/api/chat"



In [4]:
@dataclass
class AspectResult:
    aspect_id: str
    answer: str
    rationale: str


def load_gif_frames(path: Path) -> List[Image.Image]:
    """Load all frames from a GIF as PIL images."""
    gif = imageio.mimread(path)
    return [Image.fromarray(frame) for frame in gif]


def select_key_frames(frames: Sequence[Image.Image], count: int = 8) -> List[Image.Image]:
    """Uniformly sample a subset of frames for the VLM input."""
    if not frames:
        raise ValueError("GIF contains no frames")
    indices = np.linspace(0, len(frames) - 1, num=min(count, len(frames)), dtype=int)
    return [frames[i] for i in indices]


def encode_image(image: Image.Image) -> str:
    """Encode an image as data URI (base64 PNG) for Ollama's HTTP API."""
    from io import BytesIO

    buf = BytesIO()
    image.convert("RGB").save(buf, format="PNG")
    encoded = base64.b64encode(buf.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"



In [5]:
def build_prompt(aspects: Sequence[dict]) -> str:
    """Create the textual instructions for the VLM."""
    aspect_lines = [
        f"{item['id']}. {item['title']}: {item['description']}" for item in aspects
    ]
    description = """
You are an inspector judging a LunarLander-v3 attempt from a short GIF.
For each aspect (A-F) answer strictly "yes" or "no" plus a 1-2 sentence rationale citing observable evidence.
Return JSON using the schema [{"aspect": "A", "answer": "yes", "rationale": "..."}].
If an aspect cannot be determined, answer "unknown" and explain why.
"""
    return textwrap.dedent(description).strip() + "\n\nAspects:\n" + "\n".join(aspect_lines)



In [ ]:
def call_phi3_vision(prompt: str, frames: Sequence[Image.Image]) -> List[AspectResult]:
    """Send text+frames to Ollama's vision-capable model and parse JSON."""
    content = [{"type": "text", "text": prompt}]
    for frame in frames:
        content.append({
            "type": "image_url",
            "image_url": {"url": encode_image(frame)},
        })

    payload = {
        "model": OLLAMA_MODEL,  # e.g., "llava-phi3"
        "messages": [{"role": "user", "content": content}],
        "stream": False,
        "options": {"temperature": 0.2},
    }
    response = requests.post(OLLAMA_URL, json=payload, timeout=120)
    if not response.ok:
        raise RuntimeError(
            f"Ollama call failed ({response.status_code}): {response.text}"
        )
    data = response.json()
    raw_content = data.get("message", {}).get("content", "")
    try:
        parsed = json.loads(raw_content) if raw_content else []
    except json.JSONDecodeError:
        raise ValueError(f"Model returned non-JSON content: {raw_content}")

    results = []
    for item in parsed:
        results.append(
            AspectResult(
                aspect_id=item.get("aspect", ""),
                answer=item.get("answer", "unknown").lower(),
                rationale=item.get("rationale", ""),
            )
        )
    return results



In [7]:
def verify_gif(path: str | Path, frame_count: int = 8) -> List[AspectResult]:
    """Full pipeline: load GIF, sample frames, query VLM."""
    gif_path = Path(path)
    frames = load_gif_frames(gif_path)
    key_frames = select_key_frames(frames, count=frame_count)
    prompt = build_prompt(ASPECTS)
    return call_phi3_vision(prompt, key_frames)



In [9]:
# Example usage (uncomment and set a valid GIF path to run)
results = verify_gif("./lunar_gifs/lunar_lander.gif")
for item in results:
    print(f"{item.aspect_id}: {item.answer}\n  {item.rationale}\n")


HTTPError: 400 Client Error: Bad Request for url: http://localhost:11434/api/chat

## Running tips

1. Start Ollama server locally (`ollama serve`) and ensure `llava-phi3` is available.
2. Provide clips where the full descent + touchdown are visible; longer GIFs may need more sampled frames (increase `frame_count`).
3. Save raw VLM responses for auditing, especially if the output JSON schema changes.
4. Combine the aspect verdicts with simulator telemetry (reward, leg contacts) for final pass/fail logic.
